In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_baseline'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)

if not OASIS2_BUNDLE_ROOT.exists():
    raise FileNotFoundError(f'Missing uploaded OASIS-2 bundle: {OASIS2_BUNDLE_ROOT}')


Mounted at /content/drive
DRIVE_ROOT True /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT True /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [2]:
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/content/Cerebrasense-')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/Billrichard209/Cerebrasense-.git'],
    cwd='/content',
    check=True,
)

if not BACKEND_ROOT.exists():
    raise FileNotFoundError(f'Expected backend after clone: {BACKEND_ROOT}')

requirements = BACKEND_ROOT / 'requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], check=True)

repo_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
required_commit = 'a37f36a'
subprocess.run(['git', 'merge-base', '--is-ancestor', required_commit, 'HEAD'], cwd=REPO_ROOT, check=True)
print('repo_root =', REPO_ROOT)
print('backend_root =', BACKEND_ROOT)
print('repo_commit =', repo_commit)
print('required_commit =', required_commit)


repo_root = /content/Cerebrasense-
backend_root = /content/Cerebrasense-/alz_backend
repo_commit = 53910edaa811ec3d975bfb724cf7970c4d64be8a


In [3]:
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '20',
    '--batch-size', '1',
    '--gradient-accumulation-steps', '1',
    '--num-workers', '0',
    '--image-size', '64', '64', '64',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
    '--stage-bundle-to-local',
]

print('RUNNING:', ' '.join(cmd))
completed = subprocess.run(cmd, cwd=BACKEND_ROOT, text=True, capture_output=True)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {completed.returncode}')
completed


RUNNING: /usr/bin/python3 scripts/train_oasis2_colab.py --project-root /content/Cerebrasense-/alz_backend --runtime-root /content/drive/MyDrive/Cerebrasensecloud/backend_runtime --bundle-root /content/drive/MyDrive/Cerebrasensecloud/OASIS-2 --run-name oasis2_colab_baseline --epochs 20 --batch-size 1 --gradient-accumulation-steps 1 --num-workers 0 --image-size 64 64 64 --seed 42 --split-seed 42 --device auto --stage-bundle-to-local


CompletedProcess(args=['/usr/bin/python3', 'scripts/train_oasis2_colab.py', '--project-root', '/content/Cerebrasense-/alz_backend', '--runtime-root', '/content/drive/MyDrive/Cerebrasensecloud/backend_runtime', '--bundle-root', '/content/drive/MyDrive/Cerebrasensecloud/OASIS-2', '--run-name', 'oasis2_colab_baseline', '--epochs', '20', '--batch-size', '1', '--gradient-accumulation-steps', '1', '--num-workers', '0', '--image-size', '64', '64', '64', '--seed', '42', '--split-seed', '42', '--device', 'auto', '--stage-bundle-to-local'], returncode=0)

In [5]:
from pathlib import Path
import json

blocked_summary_path = RUNTIME_ROOT / 'outputs' / 'reports' / 'onboarding' / 'oasis2_colab_bundle_summary.json'
trained_summary_path = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME / 'reports' / 'colab_run_summary.json'

print('blocked_summary_path =', blocked_summary_path)
print('trained_summary_path =', trained_summary_path)

summary_path = trained_summary_path if trained_summary_path.exists() else blocked_summary_path
if not summary_path.exists():
    raise FileNotFoundError(f'Expected an OASIS-2 summary JSON, but none was found: {summary_path}')

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))

if not summary.get('training_ready', False):
    print('Training is still blocked. Inspect training_readiness_json_path and any official_demographics_import_* paths in the summary for the remaining readiness gate.')


blocked_summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/reports/onboarding/oasis2_colab_bundle_summary.json
trained_summary_path = /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline/reports/colab_run_summary.json
{
  "generated_at": "2026-04-20T15:33:07.252485+00:00",
  "bundle_root": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2",
  "runtime_data_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/data",
  "runtime_outputs_root": "/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs",
  "metadata_template_source_path": "/content/drive/MyDrive/Cerebrasensecloud/OASIS-2/backend_reference/oasis2_metadata_template.csv",
  "training_ready": false,
  "training_started": false,
  "readiness_status": "fail",
  "blocked_reason": "Remote end closed connection without response",
  "recommendations": [
    "Inspect the official demographics import step and the saved runtime reports before 